# LLM Tracker — Discovery Mode Tutorial

Discovery mode is the **inverse of coding**. Instead of giving the LLM a codebook and asking it to find those constructs, you give it a *theoretical framework* and ask it to discover whatever constructs from that framework appear in your documents.

The workflow:
1. **Discover** — the LLM reads each document through a framework (e.g. Cognitive behavioral therapy) and reports the constructs it finds, with supporting quotes.
2. **Merge** — constructs discovered across many documents are consolidated into canonical constructs, each with an LLM-synthesized name, a definition, and its constituent constructs ranked by *prototypicality*.
3. **Review** — inspect the consolidated constructs, their frequency, and sample quotes.
4. **Build a codebook** — turn the discovered constructs into a codebook you can feed to the coding pipeline (see the main tutorial).

## 1. Install

In [ ]:
!pip install "git+https://github.com/childmindresearch/llm_tracker.git#egg=llm-tracker[tutorials]"

## 2. Imports

In [ ]:
import random

import pandas as pd

from llm_tracker import AnalyzerConfig
from llm_tracker.discovery import (
    LLMTrackerDiscoverer,
    discovery_to_codebook,
    format_merged_constructs,
    merge_constructs,
)

## 3. Configuration

- `api_key` — your OpenRouter API key, **or** the path to a `.env` file containing it.
- `llm_model` — any model listed at https://openrouter.ai/models.
- `input_data` — your documents (see the note on input types below).
- `framework` — the theoretical lens to discover through. Use one of the built-in defaults or write your own string.

In [ ]:
api_key = ""
llm_model = "google/gemini-3-flash-preview"
input_data = "sample_data/reddit_autism_anxiety_depression.csv"
framework = "Cognitive behavioral therapy"  # any of the defaults, or your own string
temperature = 0  # 0 = deterministic/reproducible

config = AnalyzerConfig(
    api_key=api_key,
    model_name=llm_model,
    temperature=temperature,
)

### A note on input types (CSV vs. directory)

`discover()` accepts the **same inputs as the coding pipeline**, and figures out which you gave it:

- **A CSV file** — one document per *row*. You must tell it which column holds the text (`text_column`), and optionally which column to use as the document ID (`id_column`); without one, the row number is used. This is what the sample data below uses.
- **A directory** — one document per *file* (`.txt` or `.csv` files in a folder). Each file's name becomes its document ID. Use this when your documents are separate files, e.g. a folder of interview transcripts saved as `.txt`.

You don't call different functions for these — just pass a file path or a folder path to `discover()` and it routes automatically. (For a directory, `text_column` is ignored.)

## 4. Discover

One LLM call per document. Each document's discovered constructs (with quotes) are returned and, because we pass `output_dir`, saved to disk under `LLM_discovery_<timestamp>/discoveries/`.

In [ ]:
discoverer = LLMTrackerDiscoverer(config=config)

# CSV input -> text_column is required.
discoveries, disc_metadata, disc_errors = discoverer.discover(
    input_data,
    framework=framework,
    text_column="post",
    output_dir="LLM_discovery",
)

print(f"\nDiscovered constructs in {len(discoveries)} documents.")

**Directory input instead?** If your documents are separate `.txt` files in a folder, the call is even simpler — no `text_column` needed:

```python
discoveries, disc_metadata, disc_errors = discoverer.discover(
    "path/to/my_documents/",
    framework=framework,
    output_dir="LLM_discovery",
)
```

## 5. Merge / consolidate

The constructs discovered across all documents are consolidated into canonical constructs. One LLM call proposes the grouping, canonical names, definitions, and a **prototypicality** score (0–1) for each constituent — how central it is to the consolidated meaning (1 = the first thing that comes to mind).

In [ ]:
merged = merge_constructs(
    discoveries,
    config=config,
    framework=framework,
    output_path="merged_constructs.json",
)

# Review table: one row per constituent, grouped by canonical construct,
# sorted by prototypicality (highest first).
format_merged_constructs(merged)

If the merge prints warnings about constructs that were dropped, not placed, or given invalid scores, that is the model imperfectly following the consolidation instructions. The result is still returned for inspection — the warnings tell you exactly what to check.

## 6. Frequency of each consolidated construct

How often did each canonical construct appear — i.e. in how many documents? We map every discovered construct back to the canonical construct it was merged into, then count the distinct documents each canonical construct showed up in.

In [ ]:
# Map each constituent (original discovered name) -> its canonical construct.
constituent_to_canonical = {}
for canonical, data in merged.items():
    for c in data["constituents"]:
        constituent_to_canonical[c["name"]] = canonical

# Count, per canonical construct, how many documents it appeared in.
docs_per_construct = {canonical: set() for canonical in merged}
for doc_id, found in discoveries.items():
    for entry in found:
        canonical = constituent_to_canonical.get(entry["construct"])
        if canonical is not None:
            docs_per_construct[canonical].add(doc_id)

frequency = (
    pd.DataFrame(
        [
            {"construct": canonical, "n_documents": len(docs)}
            for canonical, docs in docs_per_construct.items()
        ]
    )
    .sort_values("n_documents", ascending=False)
    .reset_index(drop=True)
)
frequency

## 7. Bar chart of construct frequency

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(frequency))))
ax.barh(frequency["construct"], frequency["n_documents"])
ax.invert_yaxis()  # highest frequency at the top
ax.set_xlabel("Number of documents")
ax.set_title(f"Consolidated construct frequency ({framework})")
plt.tight_layout()
plt.show()

## 8. Sample quotes per construct

Two random quotes for each canonical construct, gathered from the documents where its constituents were discovered. (Quotes live in the per-document discoveries, not the merged summary, so we pull them from `discoveries`.)

In [ ]:
random.seed(0)  # reproducible sample

# Gather all quotes per canonical construct.
quotes_per_construct = {canonical: [] for canonical in merged}
for found in discoveries.values():
    for entry in found:
        canonical = constituent_to_canonical.get(entry["construct"])
        if canonical is not None:
            quotes_per_construct[canonical].extend(entry.get("quotes", []))

# Up to 2 random quotes each.
sample_rows = []
for canonical, quotes in quotes_per_construct.items():
    chosen = random.sample(quotes, min(2, len(quotes)))
    for q in chosen:
        sample_rows.append({"construct": canonical, "quote": q})

pd.DataFrame(sample_rows, columns=["construct", "quote"])

## 9. Turn the discoveries into a codebook

`discovery_to_codebook` converts the consolidated constructs into a codebook in the **same format the coding pipeline uses** — so discovery can seed a coding run. Each canonical construct becomes an entry with the definition synthesized during merging. (Examples are left empty for now; you can add them by hand, or let the codebook optimizer add them after a coding run.)

In [ ]:
import json

codebook = discovery_to_codebook(merged, name=f"{framework} (discovered)")

# Save it so the coding pipeline can load it by path.
with open("discovered_codebook.json", "w", encoding="utf-8") as f:
    json.dump(codebook, f, indent=2, ensure_ascii=False)

# Peek at the constructs and their definitions.
for name, entry in codebook["codebook"].items():
    print(f"- {name}: {entry['definition']}")

### Next: code with your discovered codebook

You now have `discovered_codebook.json` in the standard codebook format. You can feed it straight into the coding pipeline exactly as in the main tutorial — point `codebook_path` at this file and run `LLMTrackerAnalyzer`. Discovery → codebook → coding closes the loop: discover what's there, then measure it systematically.

You may want to review and edit the codebook first — tighten definitions, drop constructs you don't need, or add example quotes — since discovery is exploratory and its output benefits from a human pass before it becomes a measurement instrument.